In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, recall_score, precision_score

# --- AYARLAR ---
TEST_DATA_DIR = "data/prepared-data/test"
MODELS_ROOT_DIR = "models/pytorch"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Degerlendirme Ortami: {DEVICE}")
print(f"Test Veri Yolu: {TEST_DATA_DIR}")
print(f"Model Kok Yolu: {MODELS_ROOT_DIR}")

In [ ]:
# Klasor Isimleri -> Timm Model Kodlari Eslesmesi
MODEL_MAPPING = {
    "ResNeSt": "resnest50d",
    "ResNeXt": "resnext50_32x4d",
    "MobileNetV3": "mobilenetv3_large_100",
    "EfficientNetV2": "tf_efficientnetv2_m",
    "CVT": "cvt_13",
    "ConvNeXt": "convnext_base"
}

print("Model Mimarileri Tanimlandi:")
for k, v in MODEL_MAPPING.items():
    print(f"  - {k} -> {v}")

In [ ]:
# Normalizasyon (ImageNet Standartları)
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Dataset ve DataLoader
test_dataset = datasets.ImageFolder(TEST_DATA_DIR, test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

class_names = test_dataset.classes
y_true = test_dataset.targets # Gerçek etiketler (Liste sırası bozulmamalı)

print(f"📂 Test Seti: {len(test_dataset)} görüntü")
print(f"🏷️ Sınıflar: {class_names}")

In [ ]:
def evaluate_single_model(model_name, timm_arch, weights_path):
    print(f"\n🔍 İNCELENİYOR: {model_name} ({timm_arch})")
    print(f"   Dosya: {weights_path}")

    # 1. Modeli Boş Olarak Oluştur
    try:
        model = timm.create_model(timm_arch, pretrained=False, num_classes=NUM_CLASSES)
    except Exception as e:
        print(f"❌ HATA: Model mimarisi oluşturulamadı ({timm_arch}). Timm sürümünü veya model adını kontrol et.")
        return None

    # 2. Ağırlıkları Yükle
    try:
        model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    except Exception as e:
        print(f"❌ HATA: Ağırlıklar yüklenemedi. Dosya bozuk veya mimari uyuşmazlığı var.\nHata: {e}")
        return None

    model = model.to(DEVICE)
    model.eval()

    # 3. Tahmin Yap (Inference)
    all_preds = []

    with torch.no_grad():
        for inputs, _ in test_loader: # Etiketleri dataloader'dan değil, y_true'dan alacağız (garanti olsun)
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())

    return all_preds

In [ ]:
results_list = []

# Ana model klasöründeki alt klasörleri (ResNeXt, CVT vb.) gez
for folder_name, timm_arch in MODEL_MAPPING.items():

    # MODELS_ROOT_DIR içinde ilgili model adını içeren klasörü bul
    target_folder = None
    for d in os.listdir(MODELS_ROOT_DIR):
        if folder_name.lower() in d.lower() and os.path.isdir(os.path.join(MODELS_ROOT_DIR, d)):
            target_folder = os.path.join(MODELS_ROOT_DIR, d)
            break

    if not target_folder:
        print(f"⚠️ UYARI: '{folder_name}' için klasör bulunamadı. Atlanıyor.")
        continue

    # Klasörün içindeki 'best_model.pth' dosyasını bul
    weights_file = os.path.join(target_folder, "best_model.pth")

    if not os.path.exists(weights_file):
        print(f"⚠️ UYARI: {target_folder} içinde 'best_model.pth' yok. Atlanıyor.")
        continue

    # --- DEĞERLENDİRME YAP ---
    y_pred = evaluate_single_model(folder_name, timm_arch, weights_file)

    if y_pred is None: continue

    # --- METRİKLERİ HESAPLA ---
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    precision = precision_score(y_true, y_pred, average='weighted')

    results_list.append({
        "Model": folder_name,
        "Architecture": timm_arch,
        "Accuracy": acc,
        "F1-Score": f1,
        "Recall": recall,
        "Precision": precision,
        "Predictions": y_pred # Confusion Matrix için saklıyoruz
    })

    print(f"   ✅ Tamamlandı -> F1: {f1:.4f} | Recall: {recall:.4f}")

print("\n🏁 Tüm modeller değerlendirildi.")

In [ ]:
if len(results_list) > 0:
    df_results = pd.DataFrame(results_list)
    # Tahminleri tablodan çıkar (görsel kirlilik yapmasın)
    display_df = df_results.drop(columns=["Predictions"])

    # F1 Skoruna göre sırala (Büyükten küçüğe)
    display_df = display_df.sort_values(by="F1-Score", ascending=False)

    print("\n🏆 MODEL LİDERLİK TABLOSU 🏆")
    print(display_df.to_markdown(index=False, floatfmt=".4f"))

    # CSV olarak kaydet (proje içi sabit yol)
    output_csv = os.path.join(MODELS_ROOT_DIR, "Final_Benchmark_Results.csv")
    display_df.to_csv(output_csv, index=False)
    print(f"Sonuç CSV kaydedildi: {output_csv}")
else:
    print("Hiçbir sonuç üretilemedi.")

In [ ]:
if len(results_list) > 0:
    # Kaç model varsa ona göre grid oluştur
    num_models = len(results_list)
    cols = 2
    rows = (num_models + 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(15, 6 * rows))
    axes = axes.flatten()

    for idx, res in enumerate(results_list):
        cm = confusion_matrix(y_true, res["Predictions"])
        model_name = res["Model"]
        f1 = res["F1-Score"]

        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                    xticklabels=class_names, yticklabels=class_names)
        axes[idx].set_title(f"{model_name}\nF1: {f1:.4f}")
        axes[idx].set_xlabel("Tahmin")
        axes[idx].set_ylabel("Gerçek")

    # Boş kalan grafikleri gizle
    for i in range(idx + 1, len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    output_png = os.path.join(MODELS_ROOT_DIR, "All_Models_Confusion_Matrix.png")
    plt.savefig(output_png)
    print(f"Confusion matrix görseli kaydedildi: {output_png}")
    plt.show()